In [ ]:
# little helper so that we don't have to manually change paths everywhere
# basically defines the directory structure


# how it works (automatically retrieves home path, then concatenates everything following): 
# >>> some_path = Path.home() / "some_dir" / "some_other_dir"
# >>> some_path
# WindowsPath('C:/Users/smitz/some_dir/some_other_dir')

import re # regular expressions (magic)
from pathlib import Path

TRAINING_DIR = Path.home() / "Documents" / "github-repos" / "seals" / "_training"

DATA_CONFIG = TRAINING_DIR / "clay_config.yaml"
DATA_DIR    = TRAINING_DIR / "data"
DATASET     = DATA_DIR / "clay_seals_dataset" # change this for new training data (ALSO IN CONFIG.YAML)
TEST_IMAGE_DIR = DATASET/ "images" / "test"

RUNS_DIR = TRAINING_DIR / "runs"
OUTPUTS_DIR = TRAINING_DIR / "outputs"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)


def next_run_name(output_dir: Path, prefix: str = "model_") -> str:
    """helper to increment model output directory numbering
    (wouldn't want to have to manually change anything!)"""
    pattern = re.compile(rf"^{re.escape(prefix)}(\d+)$")

    run_numbers = [
        int(match.group(1))
        for path in output_dir.iterdir()
        if path.is_dir()
        if (match := pattern.match(path.name))
    ]

    next_number = max(run_numbers, default=0) + 1
    return f"{prefix}{next_number}"

In [13]:
# start / set up for new run

RUN_NAME = next_run_name(RUNS_DIR)

RUN_DIR = RUNS_DIR / RUN_NAME
RUN_DIR.mkdir()

MODEL_WEIGHTS = RUN_DIR / "weights" / "best.pt"

OUTPUT_DIR = OUTPUTS_DIR / RUN_NAME
DETECTIONS_DIR = OUTPUT_DIR / "detections"
DETECTIONS_CSV = OUTPUT_DIR / "detections.csv"

DETECTIONS_DIR.mkdir(parents=True)

print(f"Starting {RUN_NAME}")
print(f"Training results: {RUN_DIR}")
print(f"Inference outputs: {OUTPUT_DIR}")

Starting model_1
Training results: C:\Users\smitz\Documents\github-repos\seals\_training\runs\model_1
Inference outputs: C:\Users\smitz\Documents\github-repos\seals\_training\outputs\model_1


In [2]:
# These area libraries necessary for the code to work. The most important is the ultralytics one because it is used for model training
%pip install ultralytics
#not sure if this ever used, so try to just leave it out :)
#%pip install tqdm 
#used for inference after training and validation, basically creates a sliding window to go over images and then patches detections together
%pip install sahi 
#this one is needed to be compatible with cuda (not sure if you need this)
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130 

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 11.0 MB/s  0:00:00

   ---------------------------------------- 0/6 [tqdm]
   ---------------------------------------- 0/6 [tqdm]
   ------ --------------------------------- 1/6 [termcolor]
   ------------- -------------------------- 2/6 [shapely]
   ------------- -------------------------- 2/6 [shapely]
   ------------- -------------------------- 2/6 [shapely]
   ------------- -------------------------- 2/6 [shapely]
   ------------- -------------------------- 2/6 [shapely]
   ------------- -------------------------- 2/6 [shapely]
   ------------- -------------------------- 2/6 [shapely]
   ------------- -------------------------- 2/6 [shapely]
   ------------- -------------------------- 2/6 [shapely]
   ------------- -------------------------- 2/6 [shapely]
   ------------- -----------------------

In [ ]:
#Karl: took my poor laptop 48min!

#Model training
from ultralytics import YOLO
model = YOLO("yolov8n.pt")

# Display model information (optional)
model.info()

# Train the model on the COCO8 example dataset for 100 epochs
results = model.train(
    data = str(DATA_CONFIG), #adjust path to config file
    epochs=150, #look at https://docs.ultralytics.com/modes/train under Training settings and Augmentation setting and hyperparameters to see the list of changes that can be done to adjust a model
    patience= 50, 
    batch= 8,
    imgsz= 640,
    save= True,
    project = str(RUNS_DIR), #gives the project a name
    name = RUN_NAME, #gives the specific training run a name
    exist_ok = True,
    flipud = 0.5, #flips to image up and down
    fliplr = 0.5, #flips the images left and right
    mosaic = 0.5) #creates a mosaic of 4 different images

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\smitz\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs
New https://pypi.org/project/ultralytics/8.4.159 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.157  Python-3.14.6 torch-2.14.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=clay_config.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.

In [ ]:
#Model validation
from ultralytics import YOLO
model = YOLO(str(MODEL_WEIGHTS))

# Validate the model
metrics = model.val(
    project=str(OUTPUT_DIR),
    classes = [0],
    name = "clay_model_val1"
)  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps

Ultralytics 8.4.157  Python-3.14.6 torch-2.14.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1557.4139.3 MB/s, size: 4423.7 KB)
val: Scanning C:\Users\smitz\Documents\github-repos\seals\_misc\wetransfer\clay_seals_dataset\labels\val.cache... 11 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 11/11 2.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.8s/it 1.8s
                   all         11          9      0.888          1      0.984      0.941
                 adult          9          9      0.888          1      0.984      0.941
Speed: 3.2ms preprocess, 49.4ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to C:\Users\smitz\Documents\github-repos\seals\runs\detect\Clay_seal_model\clay_model_val1


array([    0.94063,     0.94063])

In [ ]:
import os
import csv
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel
from PIL import Image

#Paths
model = str(MODEL_WEIGHTS)  # load a custom model
image_dir = str(TEST_IMAGE_DIR) #path to test images under labels/text
output_dir = str(DETECTIONS_DIR) #path for images with detection boxes drawn
output_csv = str(DETECTIONS_CSV) #path for csv file to get raw numbers of detections for each image

os.makedirs(output_dir, exist_ok=True)

#Load model

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path= model,
    confidence_threshold=0.65,
    device="cuda:0",
)


# CSV setup
with open(output_csv, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image_name", "num_detections"])

    # Loop through images
    for image_name in os.listdir(image_dir):
        if not image_name.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        image_path = os.path.join(image_dir, image_name)

        result = get_sliced_prediction(
            image_path,
            detection_model,
            slice_height=1024,
            slice_width=1024,
            overlap_height_ratio=0.3,
            overlap_width_ratio=0.3,
        )

    
        # Filter detections by class (keep only seals, class 0)
        detections = [
            obj for obj in result.object_prediction_list
            if obj.category.id == 0
        ]

        num_detections = len(detections)

        # Write to CSV
        writer.writerow([image_name, num_detections])

        # Save image ONLY if detections exist
        if num_detections > 0:
            result.export_visuals(
                export_dir=output_dir,
                file_name=image_name,
                rect_th=1,
                text_size=0.4,
            )

Performing prediction on 24 slices.
Performing prediction on 24 slices.
Performing prediction on 24 slices.
Performing prediction on 24 slices.
Performing prediction on 24 slices.
Performing prediction on 24 slices.


In [ ]:
# Exports model to onnx format
from ultralytics import YOLO

# Load a YOLO26 model
model = YOLO(str(MODEL_WEIGHTS))

# Export the model to LiteRT format
model.export(
data = str(DATA_CONFIG),
format="onnx", 
imgsz=640
)

Ultralytics 8.4.157  Python-3.14.6 torch-2.14.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'C:\Users\smitz\Documents\github-repos\seals\runs\detect\Clay_seal_model\model_1\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (6.0 MB)

ONNX: starting export with onnx 1.23.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success  1.3s, saved as 'C:\Users\smitz\Documents\github-repos\seals\runs\detect\Clay_seal_model\model_1\weights\best.onnx' (11.7 MB)

Export complete (1.6s)
Results saved to C:\Users\smitz\Documents\github-repos\seals\runs\detect\Clay_seal_model\model_1\weights\best.onnx
Predict:         yolo predict task=detect model=C:\Users\smitz\Documents\github-repos\seals\runs\detect\Clay_seal_model\model_1\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=C:\Users\smitz\Documents\github-repos\sea

'C:\\Users\\smitz\\Documents\\github-repos\\seals\\runs\\detect\\Clay_seal_model\\model_1\\weights\\best.onnx'